# Training Passive Liveness Detection Model
Notebook ini dibuat untuk melatih model mendeteksi Liveness (Live vs Spoof).

In [1]:
import os
import cv2
import numpy as np
import gc
from mtcnn import MTCNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import pickle

detector = MTCNN()

In [2]:
def apply_clahe_and_resize(face, size=(128, 128)):
    lab = cv2.cvtColor(face, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    lab = cv2.merge((l,a,b))
    face = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    face = cv2.resize(face, size, interpolation=cv2.INTER_AREA)
    return face.astype("float32")

def detect_and_preprocess_face(img):
    h, w = img.shape[:2]
    scale = 640 / max(h, w)
    if scale < 1:
        img = cv2.resize(img, (int(w * scale), int(h * scale)))
        
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb)
    results = [r for r in results if r['confidence'] >= 0.8]
    
    if len(results) > 0:
        results = sorted(results, key=lambda x: x['box'][2] * x['box'][3], reverse=True)
        result = results[0]
        x, y, w, h = result['box']
        pad = int(min(w, h) * 0.1)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(rgb.shape[1], x+w+pad), min(rgb.shape[0], y+h+pad)
        face = rgb[y1:y2, x1:x2]
    else:
        # Jika wajah tidak terdeteksi, asumsikan gambar sudah pre-cropped
        face = rgb
        
    if face.size == 0:
        return None
        
    return apply_clahe_and_resize(face)

In [3]:
def load_dataset(base_folder):
    images = []
    labels = []
    skipped = 0
    classes = ['live', 'spoof']
    
    print(f"[INFO] Loading from {base_folder}...")
    for class_name in classes:
        folder_path = os.path.join(base_folder, class_name)
        if not os.path.isdir(folder_path):
            continue
            
        files = os.listdir(folder_path)
        used = 0
        for i, name in enumerate(files):
            if not name.lower().endswith(('.jpg', '.jpeg', '.png')):
                continue
                
            img_path = os.path.join(folder_path, name)
            img = cv2.imread(img_path)
            if img is None:
                skipped += 1
                continue
                
            try:
                processed_img = detect_and_preprocess_face(img)
            except Exception as e:
                skipped += 1
                continue
                
            if processed_img is not None:
                images.append(processed_img)
                labels.append(class_name)
                used += 1
            else:
                skipped += 1
                
            if i % 50 == 0:
                print(f"Processed {i}/{len(files)} for {class_name}", end="\r")
                gc.collect()
                
        print(f"\n[OK] {class_name}: {used} images loaded.")
        
    return np.array(images, dtype="float32"), np.array(labels)

# Memuat data train dan test
x_train, y_train_labels = load_dataset('Dataset/Dataset_liveness/train')
x_test, y_test_labels = load_dataset('Dataset/Dataset_liveness/test')

print("x_train shape :", x_train.shape)
print("x_test shape  :", x_test.shape)

[INFO] Loading from Dataset/Dataset_liveness/train...
Processed 2050/2055 for live
[OK] live: 2055 images loaded.
Processed 2500/2525 for spoof
[OK] spoof: 2525 images loaded.
[INFO] Loading from Dataset/Dataset_liveness/test...
Processed 500/514 for live
[OK] live: 514 images loaded.
Processed 600/631 for spoof
[OK] spoof: 631 images loaded.
x_train shape : (4580, 128, 128, 3)
x_test shape  : (1145, 128, 128, 3)


In [4]:
# Normalisasi data
x_train = x_train / 255.0
x_test = x_test / 255.0

# Encoding label: live -> 0, spoof -> 1
le = LabelEncoder()
y_train = le.fit_transform(y_train_labels)
y_test = le.transform(y_test_labels)

# Simpan label encoder
os.makedirs('model', exist_ok=True)
with open("model/Label_encoder_liveness.pickle", "wb") as f:
    pickle.dump(le, f)

print("Classes:", le.classes_)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Classes: ['live' 'spoof']
y_train shape: (4580,)
y_test shape: (1145,)


In [5]:
# Model CNN untuk Liveness (Binary Classification)
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(128, (3,3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid') # Sigmoid untuk binary classification (live/spoof)
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

d:\FaceAttend_VC\venv\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 126, 126, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 61, 61, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,305,665 (12.61 MB)

 Trainable params: 3,305,217 (12.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [ ]:
# Augmentasi data (hanya untuk data training)
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Callbacks
checkpoint = ModelCheckpoint('liveness_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Training
epochs = 30
batch_size = 32

history = model.fit(
    datagen.flow(x_train, y_train, batch_size=batch_size),
    validation_data=(x_test, y_test),
    epochs=epochs,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 809ms/step - accuracy: 0.6180 - loss: 1.5610
Epoch 1: val_accuracy improved from None to 0.44891, saving model to model/liveness_model.h5


144/144 ━━━━━━━━━━━━━━━━━━━━ 130s 869ms/step - accuracy: 0.6773 - loss: 0.8733 - val_accuracy: 0.4489 - val_loss: 10.8592
Epoch 2/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 788ms/step - accuracy: 0.7525 - loss: 0.5367
Epoch 2: val_accuracy improved from 0.44891 to 0.45240, saving model to model/liveness_model.h5


144/144 ━━━━━━━━━━━━━━━━━━━━ 119s 824ms/step - accuracy: 0.7511 - loss: 0.5189 - val_accuracy: 0.4524 - val_loss: 3.2140
Epoch 3/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.7871 - loss: 0.4815
Epoch 3: val_accuracy improved from 0.45240 to 0.62969, saving model to model/liveness_model.h5


144/144 ━━━━━━━━━━━━━━━━━━━━ 119s 823ms/step - accuracy: 0.7775 - loss: 0.4868 - val_accuracy: 0.6297 - val_loss: 0.6462
Epoch 4/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 750ms/step - accuracy: 0.7652 - loss: 0.4821
Epoch 4: val_accuracy did not improve from 0.62969
144/144 ━━━━━━━━━━━━━━━━━━━━ 114s 792ms/step - accuracy: 0.7886 - loss: 0.4522 - val_accuracy: 0.5983 - val_loss: 1.2721
Epoch 5/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 843ms/step - accuracy: 0.8080 - loss: 0.4389
Epoch 5: val_accuracy did not improve from 0.62969
144/144 ━━━━━━━━━━━━━━━━━━━━ 127s 884ms/step - accuracy: 0.8186 - loss: 0.4236 - val_accuracy: 0.5878 - val_loss: 1.1817
Epoch 6/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 775ms/step - accuracy: 0.8417 - loss: 0.3872
Epoch 6: val_accuracy did not improve from 0.62969
144/144 ━━━━━━━━━━━━━━━━━━━━ 117s 810ms/step - accuracy: 0.8478 - loss: 0.3763 - val_accuracy: 0.5624 - val_loss: 2.9513
Epoch 7/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 798ms/step - accuracy: 0.8590 - loss: 0.3384
Epoch 7: va

144/144 ━━━━━━━━━━━━━━━━━━━━ 147s 863ms/step - accuracy: 0.8681 - loss: 0.3194 - val_accuracy: 0.8437 - val_loss: 0.3943
Epoch 9/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 818ms/step - accuracy: 0.8599 - loss: 0.3411
Epoch 9: val_accuracy did not improve from 0.84367
144/144 ━━━━━━━━━━━━━━━━━━━━ 128s 890ms/step - accuracy: 0.8635 - loss: 0.3361 - val_accuracy: 0.5729 - val_loss: 1.6920
Epoch 10/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 792ms/step - accuracy: 0.8473 - loss: 0.3774
Epoch 10: val_accuracy did not improve from 0.84367
144/144 ━━━━━━━━━━━━━━━━━━━━ 120s 829ms/step - accuracy: 0.8410 - loss: 0.3829 - val_accuracy: 0.6882 - val_loss: 0.9986
Epoch 11/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 922ms/step - accuracy: 0.8690 - loss: 0.3105
Epoch 11: val_accuracy did not improve from 0.84367
144/144 ━━━━━━━━━━━━━━━━━━━━ 139s 966ms/step - accuracy: 0.8801 - loss: 0.2933 - val_accuracy: 0.6210 - val_loss: 1.3902
Epoch 12/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 801ms/step - accuracy: 0.8916 - loss: 0.2651
Epoch 

144/144 ━━━━━━━━━━━━━━━━━━━━ 120s 832ms/step - accuracy: 0.8902 - loss: 0.2791 - val_accuracy: 0.8672 - val_loss: 0.3802
Epoch 15/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 783ms/step - accuracy: 0.9059 - loss: 0.2365
Epoch 15: val_accuracy did not improve from 0.86725
144/144 ━━━━━━━━━━━━━━━━━━━━ 120s 833ms/step - accuracy: 0.9041 - loss: 0.2430 - val_accuracy: 0.8672 - val_loss: 0.3549
Epoch 16/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 793ms/step - accuracy: 0.9150 - loss: 0.2102
Epoch 16: val_accuracy did not improve from 0.86725
144/144 ━━━━━━━━━━━━━━━━━━━━ 120s 829ms/step - accuracy: 0.9063 - loss: 0.2325 - val_accuracy: 0.7799 - val_loss: 0.5351
Epoch 17/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 837ms/step - accuracy: 0.9155 - loss: 0.2181
Epoch 17: val_accuracy did not improve from 0.86725
144/144 ━━━━━━━━━━━━━━━━━━━━ 128s 889ms/step - accuracy: 0.9009 - loss: 0.2511 - val_accuracy: 0.5572 - val_loss: 6.0688
Epoch 18/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 801ms/step - accuracy: 0.8956 - loss: 0.2724
Epoc

144/144 ━━━━━━━━━━━━━━━━━━━━ 116s 808ms/step - accuracy: 0.9133 - loss: 0.2198 - val_accuracy: 0.8786 - val_loss: 0.3432
Epoch 21/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 781ms/step - accuracy: 0.9256 - loss: 0.1985
Epoch 21: val_accuracy improved from 0.87860 to 0.93100, saving model to model/liveness_model.h5


144/144 ━━━━━━━━━━━━━━━━━━━━ 118s 817ms/step - accuracy: 0.9275 - loss: 0.1996 - val_accuracy: 0.9310 - val_loss: 0.1672
Epoch 22/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 785ms/step - accuracy: 0.9193 - loss: 0.2059
Epoch 22: val_accuracy did not improve from 0.93100
144/144 ━━━━━━━━━━━━━━━━━━━━ 119s 827ms/step - accuracy: 0.9177 - loss: 0.2092 - val_accuracy: 0.5590 - val_loss: 4.0683
Epoch 23/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 775ms/step - accuracy: 0.9174 - loss: 0.2171
Epoch 23: val_accuracy did not improve from 0.93100
144/144 ━━━━━━━━━━━━━━━━━━━━ 117s 810ms/step - accuracy: 0.9234 - loss: 0.1905 - val_accuracy: 0.9223 - val_loss: 0.2101
Epoch 24/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - accuracy: 0.9244 - loss: 0.1805
Epoch 24: val_accuracy did not improve from 0.93100
144/144 ━━━━━━━━━━━━━━━━━━━━ 118s 820ms/step - accuracy: 0.9188 - loss: 0.1955 - val_accuracy: 0.8332 - val_loss: 0.4653
Epoch 25/30
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 808ms/step - accuracy: 0.9287 - loss: 0.1914
Epoc

### Pengujian Model secara Real-Time
Jalankan cell di bawah ini untuk membuka webcam dan menguji model yang baru saja dilatih.

In [7]:
import cv2
import numpy as np

print("[INFO] Membuka Webcam...")
cap = cv2.VideoCapture(0)

print("Tekan 'q' pada keyboard di jendela kamera untuk keluar.")

while True:
    ret, frame = cap.read()
    if not ret:
        break
        
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = detector.detect_faces(rgb_frame)
    
    for result in results:
        if result['confidence'] < 0.8:
            continue
            
        x, y, w, h = result['box']
        pad = int(min(w, h) * 0.1)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(frame.shape[1], x+w+pad), min(frame.shape[0], y+h+pad)
        
        face_crop = rgb_frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue
            
        # Preprocessing 
        processed_face = apply_clahe_and_resize(face_crop)
        processed_face = processed_face / 255.0  # Normalisasi
        processed_face = np.expand_dims(processed_face, axis=0)
        
        # Prediksi
        pred = model.predict(processed_face, verbose=0)[0][0]
        
        if pred < 0.5:
            label = "Live"
            color = (0, 255, 0)
            confidence = (1 - pred) * 100
        else:
            label = "Spoof"
            color = (0, 0, 255)
            confidence = pred * 100
            
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        text = f"{label}: {confidence:.1f}%"
        (text_width, text_height), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(frame, (x1, y1 - 25), (x1 + text_width, y1), color, -1)
        cv2.putText(frame, text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        
    cv2.imshow('Liveness Test - Tekan Q untuk keluar', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


[INFO] Membuka Webcam...
Tekan 'q' pada keyboard di jendela kamera untuk keluar.
